# ⚠️ Required First Step — Enable GPU

1. Click **Runtime** in the menu above
2. Click **Change runtime type**
3. Set **Hardware accelerator** to **T4 GPU** or **A100**
4. Click **Save**

Then run cells one by one from the top (do NOT use Run All — Drive mount needs manual auth).

---

# TunedAI Labs — Causal Reasoning Demo

This notebook compares two versions of the same model on ten causal reasoning questions:

| Model | What it is |
|---|---|
| Base Qwen 2.5-7B | Unmodified open-source model |
| **TunedAI Labs Causal Model** | Same model, fine-tuned on 10,112 causal reasoning questions |

Questions are short structured scenarios requiring YES or NO causal judgments.
Scored on correct YES/NO only — matching the format the model was trained on.

**Benchmark:** The fine-tuned model scores **96.96%** on the CLadder benchmark. Base Qwen scores ~62%.

---

---
## Step 1 — Install packages

Run this cell first. Wait for all ✓ before continuing.

In [ ]:
import subprocess, sys

pkgs = ['transformers', 'peft', 'accelerate', 'bitsandbytes', 'openai', 'anthropic']
print(f"Installing {len(pkgs)} packages...")
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs,
                   capture_output=True, text=True)
if r.returncode != 0:
    print("Warning:", r.stderr[-200:])

print("Verifying:")
ok = True
for pkg in pkgs:
    try:
        __import__(pkg if pkg != 'bitsandbytes' else 'bitsandbytes')
        print(f"  {pkg} ✓")
    except ImportError:
        print(f"  {pkg} ✗")
        ok = False
if ok:
    print("\n✓ All packages ready.")
else:
    print("\n⚠ Some packages failed — try running this cell again.")

---
## Step 2 — API keys (optional)

Leave blank to compare Base vs TunedAI only.

In [ ]:
OPENAI_API_KEY    = ""   # optional
ANTHROPIC_API_KEY = ""   # optional

---
## Step 3 — Mount Google Drive (loads the best adapter)

This step loads the correct fine-tuned adapter (v2_e1, 96.96% CLadder).
Without this, the notebook falls back to the HuggingFace version.

Complete the auth link when it appears below.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_ADAPTER = "/content/drive/MyDrive/rungs_qwen7b_feedback_v2_e1"
    if os.path.isdir(GDRIVE_ADAPTER):
        print("✓ Best adapter found (v2_e1 — 96.96% CLadder)")
    else:
        print("⚠ Adapter not found on Drive — will use HuggingFace fallback")
        print("  (Results may differ from published 96.96% benchmark)")
except Exception as e:
    print(f"Drive mount skipped: {e}")

---
## Step 4 — Load models

Downloads and loads Qwen 2.5-7B with the TunedAI Labs adapter. Takes ~90 seconds.

In [ ]:
import torch, os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import openai, anthropic

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Go to Runtime → Change runtime type → set T4 GPU.")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {vram_gb:.0f} GB")

BASE_MODEL     = "Qwen/Qwen2.5-7B-Instruct"
GDRIVE_ADAPTER = "/content/drive/MyDrive/rungs_qwen7b_feedback_v2_e1"
HF_ADAPTER     = "tunedailabs/causal-reasoning-qwen-7b"
ADAPTER_REPO   = GDRIVE_ADAPTER if os.path.isdir(GDRIVE_ADAPTER) else HF_ADAPTER
print(f"Adapter: {'Google Drive v2_e1' if ADAPTER_REPO == GDRIVE_ADAPTER else 'HuggingFace'}")

print("Loading tokenizer...", end=" ", flush=True)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
print("✓")

if vram_gb >= 20:
    print("Loading base model float16...", end=" ", flush=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.float16, device_map="cuda:0", trust_remote_code=True)
else:
    print("Loading base model 8-bit...", end=" ", flush=True)
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
print("✓")

print("Loading TunedAI Labs adapter...", end=" ", flush=True)
tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
tuned_model.eval()
print("✓")

oai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
ant_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

print("\n✓ All models ready.")

---
## Step 5 — Run comparison

This cell defines the questions and runs all comparisons. Takes ~5 minutes.

In [ ]:
import json, os, re, torch

SYSTEM = ("You are a causal reasoning expert. "
          "Answer the question with YES or NO only, then give a brief reason.")

RESULTS_FILE = "/content/tunedai_results.json"

# 10 questions: each is (scenario, correct_answer)
# Mix of Rung 1 (association), Rung 2 (intervention), Rung 3 (counterfactual)
QUESTIONS = [
    # Rung 1 — correlation vs causation
    ("In Millbrook, 12% of filtered-water drinkers get stomach illness each year. "
     "68% of unfiltered-water drinkers get stomach illness. "
     "Is drinking unfiltered water positively associated with stomach illness?",
     "yes"),
    # Rung 1 — Simpson's paradox
    ("Drug A cures 90% of mild cases and 50% of severe cases. "
     "Drug B cures 80% of mild cases and 40% of severe cases. "
     "Doctors give Drug A more often to severe cases, so Drug A's combined rate is 60% vs Drug B's 75%. "
     "Does Drug B cause better outcomes than Drug A?",
     "no"),
    # Rung 1 — common cause
    ("In coastal cities, ice cream sales and drowning deaths both peak every July. "
     "The correlation is r = 0.97. "
     "Does buying ice cream cause drowning deaths?",
     "no"),
    # Rung 1 — reverse causation
    ("Hospitals with more nurses per patient have higher death rates than hospitals with fewer nurses. "
     "Does a higher nurse-to-patient ratio cause more patient deaths?",
     "no"),
    # Rung 2 — intervention with confounding
    ("Workers who choose to use safety harnesses have a 2% injury rate. "
     "Workers who do not use harnesses have a 35% rate. "
     "Harnesses are currently only issued on high-risk jobs. "
     "If we gave harnesses to all workers regardless of job risk, "
     "would the injury rate for harness users necessarily stay at 2%?",
     "no"),
    # Rung 2 — valid intervention
    ("In a study, patients given Drug X had blood pressure drop by 20 points on average. "
     "Patients not given Drug X had no change. Drug X directly inhibits a hormone that raises blood pressure. "
     "If we give Drug X to a patient with high blood pressure, will their blood pressure decrease?",
     "yes"),
    # Rung 2 — causal chain
    ("Smoking causes chronic inflammation. Chronic inflammation causes heart disease. "
     "Does smoking cause heart disease?",
     "yes"),
    # Rung 3 — counterfactual
    ("A patient took penicillin and their bacterial infection cleared within 3 days. "
     "Penicillin is known to cause bacterial infections to clear. "
     "If the patient had not taken penicillin, would the infection have cleared within 3 days?",
     "no"),
    # Rung 3 — counterfactual
    ("A city installed streetlights in 2010. Crime fell 30% the following year. "
     "Research confirms streetlights causally reduce crime in this type of neighborhood. "
     "If the streetlights had never been installed, would crime have fallen 30%?",
     "no"),
    # Rung 2 — survivorship bias
    ("Analysts studied strategies of 500 successful companies and found they all expanded aggressively. "
     "Failed companies that expanded aggressively were not studied — they no longer exist. "
     "Does aggressive expansion cause a company to succeed?",
     "no"),
]

def _detect_yn(text):
    t = text.strip().lower()
    words = t.split()
    first = words[0].rstrip(".,!:") if words else ""
    if first == "yes": return "yes"
    if first == "no":  return "no"
    if t.startswith("yes"): return "yes"
    if t.startswith("no"):  return "no"
    # fallback: scan first 40 chars
    snippet = t[:40]
    if "yes" in snippet and "no" not in snippet: return "yes"
    if "no" in snippet and "yes" not in snippet: return "no"
    return "?"

def ask_local(question, use_adapter=True):
    if use_adapter: tuned_model.enable_adapter_layers()
    else:           tuned_model.disable_adapter_layers()
    messages = [{"role":"system","content":SYSTEM},{"role":"user","content":question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = tuned_model.generate(**inputs, max_new_tokens=150, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def ask_gpt4(q):
    if not oai_client: return "[No OpenAI key]"
    r = oai_client.chat.completions.create(model="gpt-4o",
        messages=[{"role":"system","content":SYSTEM},{"role":"user","content":q}], max_tokens=150)
    return r.choices[0].message.content.strip()

def ask_claude(q):
    if not ant_client: return "[No Anthropic key]"
    r = ant_client.messages.create(model="claude-3-5-sonnet-20241022", max_tokens=150,
        system=SYSTEM, messages=[{"role":"user","content":q}])
    return r.content[0].text.strip()

# Load saved results
results = {}
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        results = json.load(f)
    print(f"Resuming — {len(results)} of {len(QUESTIONS)} questions already done.")

SEP = "="*70; DIV = "-"*70

for i, (question, expected) in enumerate(QUESTIONS):
    key = str(i)
    if key in results:
        r = results[key]
        b = "✓" if r["base_yn"] == expected else "✗"
        t = "✓" if r["tuned_yn"] == expected else "✗"
        print(f"[Q{i+1} done — Base {b} · TunedAI {t}]")
        continue

    print(f"\n{SEP}")
    print(f"Q{i+1} of {len(QUESTIONS)}  |  Expected: {expected.upper()}")
    print(SEP)
    print(f"{question}\n")

    base_ans  = ask_local(question, use_adapter=False)
    gpt_ans   = ask_gpt4(question)
    cld_ans   = ask_claude(question)
    tuned_ans = ask_local(question, use_adapter=True)

    b_yn = _detect_yn(base_ans)
    t_yn = _detect_yn(tuned_ans)
    b_ok = b_yn == expected
    t_ok = t_yn == expected

    for label, ans in [("BASE QWEN 2.5-7B", base_ans), ("GPT-4o", gpt_ans),
                       ("CLAUDE 3.5 SONNET", cld_ans), ("TUNEDAI LABS ★", tuned_ans)]:
        print(DIV); print(f"[ {label} ]"); print(DIV); print(ans); print()

    star = " ★" if t_ok and not b_ok else (" ↓" if b_ok and not t_ok else "")
    print(f"  Base: {b_yn.upper()} ({'✓' if b_ok else '✗'})  |  TunedAI: {t_yn.upper()} ({'✓' if t_ok else '✗'}){star}")

    results[key] = {"base_ans":base_ans,"tuned_ans":tuned_ans,
                    "base_yn":b_yn,"tuned_yn":t_yn,"expected":expected}
    with open(RESULTS_FILE,"w") as f: json.dump(results,f,indent=2)

print(f"\n{SEP}")
print("All questions done.")
print(SEP)

---
## Results

Runs automatically after questions complete. Survives Colab restarts.

In [ ]:
import json, os
RESULTS_FILE = "/content/tunedai_results.json"
QUESTIONS_N  = 10

if not os.path.exists(RESULTS_FILE):
    print("No results yet — run the cell above first.")
else:
    with open(RESULTS_FILE) as f: res = json.load(f)
    n = len(res)

    b_correct = sum(1 for r in res.values() if r["base_yn"]  == r["expected"])
    t_correct = sum(1 for r in res.values() if r["tuned_yn"] == r["expected"])

    SEP = "="*70
    print(SEP)
    print("FINAL RESULTS — TunedAI Labs Causal Reasoning Demo")
    print(f"{n} causal YES/NO questions · scored on correct answer only")
    print(SEP)
    print(f"  Base Qwen 2.5-7B (untuned) : {b_correct:2d}/{n} = {100*b_correct/n:.1f}%")
    print(f"  TunedAI Labs Causal Model ★: {t_correct:2d}/{n} = {100*t_correct/n:.1f}%")
    diff = 100*t_correct/n - 100*b_correct/n
    print(f"  Improvement                : {diff:+.1f} percentage points")
    print(SEP)
    print(f"\n{'Q':<5} {'Expected':>10} {'Base':>8} {'TunedAI':>10}")
    print(f"{'----':<5} {'--------':>10} {'----':>8} {'-------':>10}")
    for i in range(n):
        k = str(i)
        if k not in res: continue
        r = res[k]
        b = "✓" if r["base_yn"]  == r["expected"] else "✗"
        t = "✓" if r["tuned_yn"] == r["expected"] else "✗"
        flag = " ★" if t=="✓" and b=="✗" else (" ↓" if b=="✓" and t=="✗" else "")
        print(f"Q{i+1:<4} {r['expected'].upper():>10} {r['base_yn'].upper():>5} {b:>3}  {r['tuned_yn'].upper():>5} {t:>3}{flag}")

---
## Try Your Own Question

The model was trained on short structured causal questions. Use this format:

**Template:**
> In [setting], [Group A] [did X]. [Group B] [did not do X].
> [Group A result]. [Group B result].
> Does [X] cause [outcome]? Answer YES or NO.

Replace the example below and run the cell.

In [ ]:
MY_QUESTION = """
Factory workers were split into two groups. Group A received ergonomic chairs. Group B kept standard chairs.
After 6 months, Group A reported 40% less back pain. Group B reported no change.
Does using an ergonomic chair cause a reduction in back pain?
"""

SYSTEM_Q = "You are a causal reasoning expert. Answer YES or NO, then give a brief reason."

base_ans  = ask_local(MY_QUESTION.strip(), use_adapter=False)
tuned_ans = ask_local(MY_QUESTION.strip(), use_adapter=True)

print(f"Question: {MY_QUESTION.strip()}\n")
print(f"[ BASE QWEN 2.5-7B ]\n{base_ans}\n")
print(f"[ TUNEDAI LABS ★ ]\n{tuned_ans}")

---
## Share Your Results

Open a [GitHub Issue](https://github.com/tunedailabs/tunedailabs/issues/new) and paste what you saw.

**TunedAI Labs** — We fine-tune open-source LLMs for real-world reasoning tasks.
[tunedailabs.com](https://tunedailabs.com)

---
## One-time Admin — Upload correct adapter to HuggingFace

Run this once after mounting Drive to permanently fix the public adapter.
Skip if already done.

In [ ]:
HF_TOKEN = ""  # paste your HuggingFace write token here

import os
GDRIVE_ADAPTER = "/content/drive/MyDrive/rungs_qwen7b_feedback_v2_e1"
HF_REPO        = "tunedailabs/causal-reasoning-qwen-7b"

if not HF_TOKEN:
    print("Add your HuggingFace write token above to upload.")
elif not os.path.isdir(GDRIVE_ADAPTER):
    print("Mount Google Drive first (Step 3).")
else:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    print(f"Uploading v2_e1 to {HF_REPO}...")
    for fname in os.listdir(GDRIVE_ADAPTER):
        fpath = os.path.join(GDRIVE_ADAPTER, fname)
        if os.path.isfile(fpath):
            print(f"  {fname}...", end=" ", flush=True)
            api.upload_file(path_or_fileobj=fpath, path_in_repo=fname, repo_id=HF_REPO)
            print("✓")
    print("\nDone — HuggingFace now has the correct adapter (v2_e1, 96.96%)")